# Cartpole Continuous Fitted Value Iteration Only

This notebook is a cleaned-up cartpole-only version of the continuous fitted value iteration code. It includes three practical modifications for the cartpole swing-up experiment:

1. The target state is explicitly added to the training set.
2. The value function is anchored with repeated samples enforcing \(J(x_{goal}) = 0\).
3. An optional terminal-shaping cost can be added to strengthen the gradient toward the upright state.

State convention used below:

\[
x = [q, \theta, \dot q, \dot\theta]^T,
\qquad
x_{goal} = [0, \pi, 0, 0]^T.
\]

In [ ]:
import numpy as np
from functools import partial

from pydrake.all import (
    AddMultibodyPlantSceneGraph,
    DiagramBuilder,
    LeafSystem,
    MeshcatVisualizer,
    MultilayerPerceptron,
    Parser,
    PerceptronActivationType,
    RandomGenerator,
    Simulator,
    StartMeshcat,
    ZeroOrderHold,
)

from underactuated import ConfigureParser, running_as_notebook
from underactuated.optimizers import Adam
from underactuated.utils import running_as_test

In [ ]:
# Start MeshCat. Run this once per notebook kernel.
meshcat = StartMeshcat()

## Helper functions

The cost helper below uses batched column-major state samples:

\[
X = [x_1, x_2, \ldots, x_N] \in \mathbb R^{n_x \times N}.
\]

For cartpole, the pole angle is periodic, so the angle error is wrapped before evaluating the quadratic cost.

In [ ]:
def compute_quadratic_cost(Q, data):
    """Return x.T Q x for each column sample in data."""
    if len(data.shape) != 2:
        data = data.reshape(-1, 1)

    assert Q.shape[0] == data.shape[0]
    assert Q.shape[1] == data.shape[0]
    return np.sum(data * (Q @ data), axis=0)


def wrap_to_pi(a):
    """Wrap angles to [-pi, pi]."""
    return np.arctan2(np.sin(a), np.cos(a))


def compute_state_cost_wrapped(Q, target_state, data, angle_indices=()):
    """Return (x - x_goal).T Q (x - x_goal) for each column sample.

    Angle-coordinate errors are wrapped before applying the quadratic form.
    """
    if len(data.shape) != 2:
        data = data.reshape(-1, 1)

    target_state = np.asarray(target_state).reshape(data.shape[0], 1)
    err = data - target_state
    err = err.copy()

    for idx in angle_indices:
        err[idx, :] = wrap_to_pi(err[idx, :])

    return compute_quadratic_cost(Q, err)


def compute_u_star(R_diag, dJdX, dstate_dynamics_du):
    """Compute the HJB greedy input.

    Continuous-time control-affine dynamics:
        xdot = f1(x) + f2(x) u

    HJB minimizer with input cost u.T R u:
        u_star = -0.5 R^{-1} f2(x).T grad J(x)

    Args:
        R_diag: shape (num_inputs,), diagonal entries of R
        dJdX: shape (num_states, num_samples)
        dstate_dynamics_du: shape (num_states, num_inputs, num_samples)

    Returns:
        u_star: shape (num_inputs, num_samples)
    """
    assert dJdX.shape[1] == dstate_dynamics_du.shape[2]
    assert dJdX.shape[0] == dstate_dynamics_du.shape[0]
    assert R_diag.shape == (dstate_dynamics_du.shape[1],)

    # For each sample n and input m:
    #   djdu[m, n] = sum_s dJdX[s, n] * dstate_dynamics_du[s, m, n]
    # This is f2(x_n).T grad J(x_n).
    djdu = np.einsum("sn,smn->mn", dJdX, dstate_dynamics_du)
    return -0.5 * djdu / R_diag.reshape(-1, 1)

## Modified continuous fitted value iteration

This version fixes the target-state insertion bug and adds two optional stabilizers:

- `goal_anchor_weight`: repeatedly adds the goal state to each minibatch and forces its target value to zero.
- `terminal_cost_function` and `terminal_weight`: optionally add a small shaping term on the next state.

The core Bellman target is still

\[
J_d(x) = h\,\ell(x,u^*) + \gamma J_i(x + h f(x,u^*)).
\]

In [ ]:
def ContinuousFittedValueIteration(
    plant,
    plant_context,
    value_mlp,
    state_cost_function,
    compute_u_star,
    R_diag,
    state_samples,
    time_step=0.01,
    discount_factor=1.0,
    input_port_index=0,
    lr=0.001,
    minibatch=None,
    epochs=1000,
    optim_steps_per_epoch=25,
    input_limits=None,
    target_state=None,
    goal_anchor_weight=0,
    terminal_cost_function=None,
    terminal_weight=0.0,
):
    input_port = plant.get_input_port(input_port_index)
    num_states = plant.num_continuous_states()
    num_inputs = input_port.size()

    state_samples = np.asarray(state_samples)

    # Important: actually append the target state, and make it the final column.
    if target_state is not None:
        target_state = np.asarray(target_state).reshape(num_states, 1)
        if state_samples.shape[1] == 0 or not np.allclose(
            state_samples[:, -1:], target_state
        ):
            state_samples = np.hstack([state_samples, target_state])

    N = state_samples.shape[1]

    assert plant_context.has_only_continuous_state()
    assert value_mlp.get_input_port().size() == num_states
    assert value_mlp.layers()[-1] == 1
    assert R_diag.shape == (num_inputs,)
    assert state_samples.shape[0] == num_states
    assert time_step > 0.0
    assert discount_factor > 0.0 and discount_factor <= 1.0
    if input_limits is not None:
        assert num_inputs == 1, "Input limits are only supported for scalar inputs here."
        assert len(input_limits) == 2
    if goal_anchor_weight > 0:
        assert target_state is not None, "goal_anchor_weight requires target_state."

    generator = RandomGenerator(123)
    np.random.seed(123)

    mlp_context = value_mlp.CreateDefaultContext()
    value_mlp.SetRandomContext(mlp_context, generator)

    # Precompute state cost and control-affine dynamics on the sampled states.
    state_cost = state_cost_function(state_samples)
    state_dynamics_x = np.empty((N, num_states))
    dstate_dynamics_du = np.empty((num_states, num_inputs, N))
    state = plant_context.get_mutable_continuous_state_vector()

    for i in range(N):
        u = np.zeros(num_inputs)
        input_port.FixValue(plant_context, u)
        state.SetFromVector(state_samples[:, i])
        state_dynamics_x[i] = plant.EvalTimeDerivatives(plant_context).CopyToVector()

        for j in range(num_inputs):
            u[j] = 1.0
            input_port.FixValue(plant_context, u)
            dstate_dynamics_du[:, j, i] = (
                plant.EvalTimeDerivatives(plant_context).CopyToVector()
                - state_dynamics_x[i]
            )
            u[j] = 0.0

    optimizer = Adam(value_mlp.GetMutableParameters(mlp_context), lr=lr)

    if minibatch and target_state is not None:
        M = minibatch + goal_anchor_weight
    elif minibatch:
        M = minibatch
    else:
        M = N

    J = np.zeros((1, M))
    Jnext = np.zeros((1, M))
    Jd = np.zeros((1, M))
    dJdX = np.asfortranarray(np.zeros((num_states, M)))
    dloss_dparams = np.zeros(value_mlp.num_parameters())

    last_loss = np.inf
    epochs_to_run = 2 if running_as_test else epochs
    optim_steps_to_run = 2 if running_as_test else optim_steps_per_epoch

    for epoch in range(epochs_to_run):
        if minibatch:
            batch = np.random.randint(0, N, minibatch)
            if target_state is not None and goal_anchor_weight > 0:
                goal_indices = np.full(goal_anchor_weight, N - 1, dtype=int)
                batch = np.concatenate([batch, goal_indices])
        else:
            batch = np.arange(N)

        # Current value and value gradient at sampled states.
        value_mlp.BatchOutput(mlp_context, state_samples[:, batch], J, dJdX)

        # HJB greedy input.
        u_star = compute_u_star(
            R_diag,
            dJdX,
            dstate_dynamics_du[:, :, batch],
        )

        if input_limits is not None:
            u_star = np.clip(u_star, input_limits[0], input_limits[1])

        # Forward Euler transition: x_next = x + h(f1(x) + f2(x)u).
        Xdot = state_dynamics_x[batch, :].T + np.einsum(
            "smn,mn->sn",
            dstate_dynamics_du[:, :, batch],
            u_star,
        )
        Xnext = state_samples[:, batch] + time_step * Xdot

        # Running cost: l(x, u) = l_x(x) + u.T R u.
        Cost = state_cost[batch] + np.sum(
            R_diag.reshape(-1, 1) * u_star**2,
            axis=0,
        )

        # Bootstrap from current value network at x_next.
        value_mlp.BatchOutput(mlp_context, Xnext, Jnext)

        # Bellman target.
        Jd[:] = time_step * Cost.reshape(1, -1) + discount_factor * Jnext

        # Optional terminal shaping on x_next.
        if terminal_cost_function is not None and terminal_weight > 0.0:
            terminal_cost_next = terminal_cost_function(Xnext).reshape(1, -1)
            Jd[:] += terminal_weight * terminal_cost_next

        # Strongly anchor the goal value to zero.
        if target_state is not None and goal_anchor_weight > 0:
            Jd[:, -goal_anchor_weight:] = 0.0

        for _ in range(optim_steps_to_run):
            loss = value_mlp.BackpropagationMeanSquaredError(
                mlp_context,
                state_samples[:, batch],
                Jd,
                dloss_dparams,
            )
            optimizer.step(loss, dloss_dparams)

        if not minibatch and np.linalg.norm(last_loss - loss) < 1e-8:
            break
        last_loss = loss
        if epoch % 10 == 0 or epoch == epochs_to_run - 1:
            print(f"epoch {epoch}: loss = {loss}")

    return mlp_context

## Build the cartpole plant

In [ ]:
time_step = 0.01

builder = DiagramBuilder()
cart_plant, cart_scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=0.0)
parser = Parser(cart_plant)
ConfigureParser(parser)
parser.AddModelsFromUrl("package://underactuated/models/cartpole.urdf")
cart_plant.Finalize()
cart_plant_context = cart_plant.CreateDefaultContext()
cart_diagram = builder.Build()

num_states = cart_plant.num_continuous_states()
cart_actuation_port_index = cart_plant.get_actuation_input_port().get_index()
num_inputs = cart_plant.get_input_port(cart_actuation_port_index).size()

print("num_states:", num_states)
print("num_inputs:", num_inputs)
print("actuation input port index:", cart_actuation_port_index)

## Cartpole training data and costs

Start with a moderate grid for debugging. Increasing this grid may improve the learned value landscape, but it also makes the dynamics precomputation much slower.

In [ ]:
# State convention: [cart position, pole angle, cart velocity, pole angular velocity].
cart_target_state = np.array([0.0, np.pi, 0.0, 0.0]).reshape(-1, 1)

x_states_cart = np.linspace(-2.0, 2.0, 5)
theta_states_cart = np.linspace(0.0, 2.0 * np.pi, 81)
x_dot_states_cart = np.linspace(-5.0, 5.0, 5)
theta_dot_states_cart = np.linspace(-8.0, 8.0, 5)

state_grid_cart = np.meshgrid(
    x_states_cart,
    theta_states_cart,
    x_dot_states_cart,
    theta_dot_states_cart,
    indexing="ij",
)
state_data_cart = np.vstack([s.flatten() for s in state_grid_cart])

# Explicitly include the goal state as the last column.
state_data_cart = np.hstack([state_data_cart, cart_target_state])
print("number of cartpole training samples:", state_data_cart.shape[1])

# Running cost: focus strongly on upright pole angle, but do not make cart position too expensive.
Q_cart = np.diag([0.1, 100.0, 1.0, 1.0])
R_cart = np.array([0.1])

state_cost_function_cart = partial(
    compute_state_cost_wrapped,
    Q_cart,
    cart_target_state,
    angle_indices=[1],
)

# Optional terminal shaping. Keep terminal_weight small.
Qf_cart = np.diag([1.0, 300.0, 5.0, 5.0])
terminal_cost_function_cart = partial(
    compute_state_cost_wrapped,
    Qf_cart,
    cart_target_state,
    angle_indices=[1],
)

## Cartpole value network

Only the pole angle is periodic, so the input-periodicity flags are `[False, True, False, False]`. The first layer should have 5 input features because \(\theta\) is expanded to \((\cos\theta, \sin\theta)\).

In [ ]:
cart_value_mlp = MultilayerPerceptron(
    [False, True, False, False],
    [128, 128, 1],
    [
        PerceptronActivationType.kReLU,
        PerceptronActivationType.kReLU,
        PerceptronActivationType.kIdentity,
    ],
)

_tmp_context = cart_value_mlp.CreateDefaultContext()
cart_value_mlp.SetRandomContext(_tmp_context, RandomGenerator(152))
print("layer 0 weight shape:", cart_value_mlp.GetWeights(_tmp_context, 0).shape)
print("layer 1 weight shape:", cart_value_mlp.GetWeights(_tmp_context, 1).shape)
print("layer 2 weight shape:", cart_value_mlp.GetWeights(_tmp_context, 2).shape)

## Train cartpole value network

The defaults below are meant as a robust starting point, not as a guaranteed perfect swing-up. Important knobs:

- `goal_anchor_weight`: increase if the goal value is not close to zero.
- `terminal_weight`: start small; too large can blow up the target scale.
- `R_cart`: smaller values make the controller more willing to use cart force.

In [ ]:
cart_input_limits = [-20.0, 20.0]

cart_value_mlp_context = ContinuousFittedValueIteration(
    cart_plant,
    cart_plant_context,
    cart_value_mlp,
    state_cost_function_cart,
    compute_u_star,
    R_cart,
    state_data_cart,
    time_step=time_step,
    discount_factor=0.999,
    input_port_index=cart_actuation_port_index,
    lr=1e-4,
    minibatch=128,
    epochs=500,
    optim_steps_per_epoch=100,
    input_limits=cart_input_limits,
    target_state=cart_target_state,
    goal_anchor_weight=32,
    terminal_cost_function=terminal_cost_function_cart,
    terminal_weight=0.001,
)

## Policy system from the learned value function

The controller computes

\[
u^*(x) = -\frac12 R^{-1} f_2(x)^T \nabla_x J(x)
\]

at every simulation step.

In [ ]:
class ContinuousFittedValueIterationPolicyComputeUStar(LeafSystem):
    def __init__(
        self,
        plant,
        value_mlp,
        value_mlp_context,
        R_diag,
        compute_u_star,
        input_port_index=0,
        input_limits=None,
        print_debug=False,
    ):
        LeafSystem.__init__(self)

        self.num_plant_states = value_mlp.get_input_port().size()
        self._plant = plant
        self._plant_context = plant.CreateDefaultContext()
        self.value_mlp = value_mlp
        self.value_mlp_context = value_mlp_context
        self.J = np.zeros((1, 1))
        self.dJdX = np.asfortranarray(np.zeros((self.num_plant_states, 1)))
        self.compute_u_star = compute_u_star
        self.R_diag = R_diag
        self.input_limits = input_limits
        self.print_debug = print_debug

        self.DeclareVectorInputPort("plant_state", self.num_plant_states)
        self._plant_input_port = self._plant.get_input_port(input_port_index)
        self.DeclareVectorOutputPort(
            "output",
            self._plant_input_port.size(),
            self.CalcOutput,
        )

    def CalcOutput(self, context, output):
        num_inputs = self._plant_input_port.size()
        plant_state = self.get_input_port().Eval(context)

        self.value_mlp.BatchOutput(
            self.value_mlp_context,
            np.atleast_2d(plant_state).T,
            self.J,
            self.dJdX,
        )

        u = np.zeros(num_inputs)
        self._plant_context.SetContinuousState(plant_state)
        self._plant_input_port.FixValue(self._plant_context, u)
        state_dynamics_x = self._plant.EvalTimeDerivatives(
            self._plant_context
        ).CopyToVector()

        dstate_dynamics_du = np.empty((self.num_plant_states, num_inputs, 1))
        for j in range(num_inputs):
            u[j] = 1.0
            self._plant_input_port.FixValue(self._plant_context, u)
            dstate_dynamics_du[:, j, 0] = (
                self._plant.EvalTimeDerivatives(self._plant_context).CopyToVector()
                - state_dynamics_x
            )
            u[j] = 0.0

        u_star = self.compute_u_star(self.R_diag, self.dJdX, dstate_dynamics_du)[:, 0]
        if self.input_limits is not None:
            u_star = np.clip(u_star, self.input_limits[0], self.input_limits[1])

        if self.print_debug:
            print(
                "state:", plant_state,
                "J:", self.J.ravel(),
                "dJdX:", self.dJdX.ravel(),
                "u:", u_star,
            )

        for j in range(num_inputs):
            output.SetAtIndex(j, u_star[j])

## Quick controller sanity check

This does not simulate; it only evaluates the learned policy at several states.

In [ ]:
def evaluate_cartpole_policy_at_states(states, print_debug=False):
    temp_controller = ContinuousFittedValueIterationPolicyComputeUStar(
        cart_plant,
        cart_value_mlp,
        cart_value_mlp_context,
        R_cart,
        compute_u_star,
        input_port_index=cart_actuation_port_index,
        input_limits=cart_input_limits,
        print_debug=print_debug,
    )
    temp_context = temp_controller.CreateDefaultContext()
    output = temp_controller.AllocateOutput()

    for s in states:
        temp_controller.get_input_port().FixValue(temp_context, np.asarray(s))
        temp_controller.CalcOutput(temp_context, output.get_mutable_vector_data(0))
        u = output.get_vector_data(0).CopyToVector()
        print("state =", np.asarray(s), "  u* =", u)


evaluate_cartpole_policy_at_states([
    [0.0, 0.0, 0.0, 0.0],
    [0.0, 0.2, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.5],
    [0.0, np.pi / 2.0, 0.0, 0.0],
    [0.0, np.pi, 0.0, 0.0],
])

## Closed-loop cartpole simulation

For debugging, start from a slightly perturbed downward state. Once the behavior looks active, try `[0, 0, 0, 0]`.

In [ ]:
closed_loop_builder_cart = DiagramBuilder()

cart_plant_cl, cart_scene_graph_cl = AddMultibodyPlantSceneGraph(
    closed_loop_builder_cart,
    time_step=0.0,
)
parser = Parser(cart_plant_cl)
ConfigureParser(parser)
parser.AddModelsFromUrl("package://underactuated/models/cartpole.urdf")
cart_plant_cl.Finalize()

cart_controller_sys = ContinuousFittedValueIterationPolicyComputeUStar(
    cart_plant_cl,
    cart_value_mlp,
    cart_value_mlp_context,
    R_cart,
    compute_u_star,
    input_port_index=cart_actuation_port_index,
    input_limits=cart_input_limits,
)

cart_controller = closed_loop_builder_cart.AddSystem(cart_controller_sys)
zoh_cart = closed_loop_builder_cart.AddSystem(ZeroOrderHold(time_step, num_inputs))

closed_loop_builder_cart.Connect(
    cart_plant_cl.get_state_output_port(),
    cart_controller.get_input_port(),
)
closed_loop_builder_cart.Connect(
    cart_controller.get_output_port(),
    zoh_cart.get_input_port(),
)
closed_loop_builder_cart.Connect(
    zoh_cart.get_output_port(),
    cart_plant_cl.get_input_port(cart_actuation_port_index),
)

meshcat.Delete()
meshcat.Set2dRenderMode(xmin=-2.5, xmax=2.5, ymin=-1.0, ymax=2.5)
vis = MeshcatVisualizer.AddToBuilder(
    closed_loop_builder_cart,
    cart_scene_graph_cl,
    meshcat,
)

cart_diagram_closed_loop = closed_loop_builder_cart.Build()
cart_simulator = Simulator(cart_diagram_closed_loop)
cart_simulator_context = cart_simulator.get_mutable_context()

In [ ]:
cart_simulator.set_target_realtime_rate(1.0 if running_as_notebook else 0.0)
duration = 0.1 if running_as_test else 10.0

# Debug initial conditions. Try these one by one.
initial_state = [0.0, 0.2, 0.0, 0.0]
# initial_state = [0.0, 0.0, 0.0, 0.5]
# initial_state = [0.0, 0.0, 0.0, 0.0]

cart_simulator_context.SetTime(0.0)
cart_simulator_context.SetContinuousState(initial_state)
cart_simulator.Initialize()
cart_simulator.AdvanceTo(duration)